# N06 — Figure 4: thermal no-crossing theorem and nonequilibrium Gaussian crossings

This notebook is the theoretical companion to the manuscript's Ornstein–Uhlenbeck section. It does more than redraw Figure 4: it derives the mean/covariance propagation, verifies the stationary Gibbs covariance, demonstrates why centered Gibbs preparations **cannot** cross, and then shows how nonequilibrium structure in either the mean or covariance restores the possibility of a crossing.

The thermal OU model is an ideal equilibrium extension. It is deliberately kept separate from the engineered-noise laboratory experiment of N05.


In [ ]:
from pathlib import Path
import json, csv, re, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy.linalg import expm

# Locate the reproducibility-bundle root whether the notebook is launched
# from the bundle root or from notebooks/.
def locate_root(start=Path.cwd()):
    candidates = [start, start.parent, start.parent.parent]
    for p in candidates:
        if (p / "experimental_data" / "raw").exists():
            return p.resolve()
    raise FileNotFoundError("Could not locate bundle root containing experimental_data/raw/." )

ROOT = locate_root()
RAW = ROOT / "experimental_data" / "raw"
CERT = ROOT / "experimental_data" / "certified"
OUT_DATA = ROOT / "notebook_outputs" / "data"
OUT_FIG = ROOT / "notebook_outputs" / "figures"
OUT_DATA.mkdir(parents=True, exist_ok=True)
OUT_FIG.mkdir(parents=True, exist_ok=True)

# ---------------- Plot-style switch ----------------
# Safe default: executes everywhere with Matplotlib only.
# Local options:
#   PLOT_STYLE = "science-no-latex"  -> requires `pip install scienceplots`
#   PLOT_STYLE = "science-latex"     -> additionally requires a local LaTeX install
PLOT_STYLE = "matplotlib"

def apply_plot_style(style=PLOT_STYLE):
    plt.style.use("default")
    if style.startswith("science"):
        try:
            import scienceplots  # noqa: F401
            plt.style.use(["science", "no-latex"] if style == "science-no-latex" else ["science"])
        except ImportError:
            warnings.warn("scienceplots is not installed; falling back to Matplotlib.")
    mpl.rcParams.update({
        "font.family": "serif",
        "font.serif": ["STIXGeneral", "DejaVu Serif"],
        "mathtext.fontset": "stix",
        "font.size": 9.0,
        "axes.labelsize": 9.0,
        "axes.titlesize": 9.5,
        "legend.fontsize": 7.5,
        "xtick.labelsize": 8.0,
        "ytick.labelsize": 8.0,
        "axes.linewidth": 0.7,
        "lines.linewidth": 1.3,
        "xtick.direction": "out",
        "ytick.direction": "out",
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",
        "savefig.bbox": "tight",
        "savefig.pad_inches": 0.04,
    })

apply_plot_style()

# Measured/component values used in the manuscript.
L_H = 44.4e-3
C_F = 454e-6
R_EXT_OHM = 22.2
R_COMP_OHM = 27.9
DETERMINISTIC_RELEASE_S = 7.2e-3

print(f"Bundle root: {ROOT}")
print(f"Plot style: {PLOT_STYLE}")


## 1. Thermal coordinates and OU drift

Define
\[
u=\frac{q}{\sqrt{Ck_BT_b}},\qquad
v=\sqrt{\frac{L}{k_BT_b}}I.
\]
In these coordinates the bath covariance is the identity and the dimensionless energy is
\[
\frac{E}{k_BT_b}=\frac{u^2+v^2}{2}.
\]
The drift matrix becomes
\[
A_T=\begin{pmatrix}0&\omega_0\\-\omega_0&-\gamma\end{pmatrix}.
\]
For the pedagogical reference spectrum we use \(\lambda_s=-100\,\mathrm{s}^{-1}\) and \(\lambda_f=-1000\,\mathrm{s}^{-1}\).


In [ ]:
lambda_s=-100.0
lambda_f=-1000.0
omega0=np.sqrt(lambda_s*lambda_f)
gamma=-(lambda_s+lambda_f)
A=np.array([[0.0,omega0],[-omega0,-gamma]])
Sigma_b=np.eye(2)
print("eigenvalues:",np.linalg.eigvals(A))
assert np.allclose(np.sort(np.linalg.eigvals(A)),np.sort([lambda_s,lambda_f]))


## 2. Exact covariance propagation and stationary-bath check

For an OU process with stationary covariance \(\Sigma_b\),
\[
\Sigma(t)=\Sigma_b+\Phi(t)[\Sigma(0)-\Sigma_b]\Phi(t)^T,
\qquad \Phi(t)=e^{At}.
\]
For a finite step \(\Delta t\), the exact noise covariance is
\[
\Sigma_\eta=\Sigma_b-\Phi\Sigma_b\Phi^T.
\]
The next cell verifies stationarity and positive semidefiniteness numerically.


In [ ]:
dt=2e-4
Phi=expm(A*dt)
Sigma_eta=Sigma_b-Phi@Sigma_b@Phi.T
stationarity=Phi@Sigma_b@Phi.T+Sigma_eta
print("stationarity residual",np.max(np.abs(stationarity-Sigma_b)))
print("Sigma_eta eigenvalues",np.linalg.eigvalsh(Sigma_eta))
assert np.max(np.abs(stationarity-Sigma_b))<1e-14
assert np.linalg.eigvalsh(Sigma_eta).min()>-1e-14


## 3. Centered Gibbs states: why crossing is impossible

A centered Gibbs preparation at temperature \(T_i\) has \(\Sigma(0)=(T_i/T_b)I\). Therefore
\[
\frac{\langle E_T(t)\rangle}{k_BT_b}
=1+\left(\frac{T_i}{T_b}-1\right)f(t),
\]
with
\[
f(t)=\frac12\operatorname{Tr}[\Phi(t)\Phi(t)^T]>0.
\]
Every initial temperature multiplies the **same positive relaxation function**. The absence of a crossing is therefore a theorem for this preparation family, not a special parameter choice.


In [ ]:
t=np.linspace(0,.012,1000);tm=1e3*t
Phi_t=np.array([expm(A*tt) for tt in t])
f=np.array([.5*np.trace(P@P.T) for P in Phi_t])
T_ratios=[1.5,2,4,7,11]
Eg={r:1+(r-1)*f for r in T_ratios}
assert np.all(f>0)
for a,b in zip(T_ratios[:-1],T_ratios[1:]):
    assert np.all(Eg[b]>Eg[a])
print("minimum f(t) over plotted interval =",f.min())


## 4. Nonequilibrium information in the mean

A displaced thermal state has the bath covariance \(\Sigma_b\) but a nonzero mean. If the mean is aligned with a drift eigenvector, the excess energy follows a pure modal exponential. We compare a lower-energy slow displacement with a higher-energy fast displacement.


In [ ]:
def unit_eigvec(lam):
    # In thermal coordinates: v = (1, lambda/omega0), normalized.
    x=np.array([1.0,lam/omega0]);return x/np.linalg.norm(x)
rs=unit_eigvec(lambda_s);rf=unit_eigvec(lambda_f)

# Excess energies at t=0: slow=1, fast=10 in units kBTb.
mu_s0=np.sqrt(2)*rs
mu_f0=np.sqrt(20)*rf

def mean_excess(mu0):
    mus=np.einsum("tij,j->ti",Phi_t,mu0)
    return .5*np.sum(mus**2,axis=1)
Dmu_s=mean_excess(mu_s0);Dmu_f=mean_excess(mu_f0)


## 5. Nonequilibrium information in the covariance

Now set the mean to zero and store the same modal excess in the covariance:
\[
\Sigma(0)=I+a_j r_jr_j^T.
\]
For \(a_s=2\) and \(a_f=20\), the initial excess energies are again 1 and 10. The total excess-energy curves are intentionally identical to the displaced case, even though the nonequilibrium information lives in a different statistical moment.


In [ ]:
Sig_s0=np.eye(2)+2*np.outer(rs,rs)
Sig_f0=np.eye(2)+20*np.outer(rf,rf)

def covariance_excess(S0):
    D0=S0-np.eye(2)
    return np.array([.5*np.trace(P@D0@P.T) for P in Phi_t])
Dsig_s=covariance_excess(Sig_s0);Dsig_f=covariance_excess(Sig_f0)
print("max difference between matched mean/covariance examples:",np.max(np.abs(Dmu_s-Dsig_s)),np.max(np.abs(Dmu_f-Dsig_f)))
assert np.allclose(Dmu_s,Dsig_s,rtol=1e-12,atol=1e-12)
assert np.allclose(Dmu_f,Dsig_f,rtol=1e-12,atol=1e-12)


## 6. Crossing time

For the matched pure-mode excess energies,
\[
\Delta E_s(t)=e^{2\lambda_s t},\qquad
\Delta E_f(t)=10e^{2\lambda_f t},
\]
so
\[
t_\times=\frac{\ln(1/10)}{2(\lambda_f-\lambda_s)}.
\]


In [ ]:
tx=np.log(1/10)/(2*(lambda_f-lambda_s));cross=np.exp(2*lambda_s*tx)
print(f"t_cross = {1e3*tx:.6f} ms, excess energy at crossing = {cross:.6f} kBTb")


## 7. Regenerate Figure 4

The 2×2 layout separates the theorem from its counterexamples:

- **(a)** several physical Gibbs temperatures remain ordered;
- **(b)** normalization exposes their common positive relaxation law;
- **(c)** a displacement stores nonequilibrium spectral weight in the first moment;
- **(d)** anisotropic covariance stores the corresponding spectral weight in the second moment.

For panels (c,d) we subtract the common bath equilibrium energy and use a logarithmic scale. This changes neither the crossing condition nor the ordering; it only exposes the relaxing spectral contribution cleanly.


In [ ]:
fig=plt.figure(figsize=(7.15,5.25))
gs=fig.add_gridspec(2,2,wspace=.31,hspace=.38)
ax=[fig.add_subplot(gs[i,j]) for i in range(2) for j in range(2)]

def panel(a,label): a.text(-.14,1.06,label,transform=a.transAxes,fontweight="bold",fontsize=11,va="bottom")

# (a)
for r in T_ratios: ax[0].plot(tm,Eg[r],label=fr"$T_i/T_b={r:g}$")
ax[0].set_xlim(0,12);ax[0].set_ylim(.9,11.35);ax[0].set_xlabel("time (ms)");ax[0].set_ylabel(r"$\langle E\rangle/(k_BT_b)$")
ax[0].set_title("Centered Gibbs states");ax[0].legend(frameon=False,fontsize=6.8);panel(ax[0],"(a)")

# (b)
ax[1].plot(tm,f,label=r"$f(t)$")
markers=["o","s","^","D","v"]
idx=np.arange(40,len(tm),180)
for r,mk,off in zip(T_ratios,markers,range(len(T_ratios))):
    ii=idx+8*off;ii=ii[ii<len(tm)]
    norm=(Eg[r]-1)/(r-1)
    ax[1].plot(tm[ii],norm[ii],ls="none",marker=mk,ms=4,label=fr"$T_i/T_b={r:g}$")
ax[1].axhline(0,ls=":",lw=.8);ax[1].set_xlim(0,12);ax[1].set_ylim(-.02,1.04)
ax[1].set_xlabel("time (ms)");ax[1].set_ylabel(r"$\Delta E_T(t)/\Delta E_T(0)$")
ax[1].set_title("Universal Gibbs relaxation");ax[1].legend(frameon=False,ncol=2,fontsize=6.5);panel(ax[1],"(b)")

# (c,d)
for a,fast_curve,slow_curve,title,label in [
    (ax[2],Dmu_f,Dmu_s,"Displaced thermal states","(c)"),
    (ax[3],Dsig_f,Dsig_s,"Centered anisotropic Gaussian states","(d)"),
]:
    a.plot(tm,fast_curve,label="fast");a.plot(tm,slow_curve,label="slow")
    a.axvline(1e3*tx,ls="--",lw=.9,alpha=.7);a.plot(1e3*tx,cross,"o",ms=4)
    a.text(1e3*tx+.12,cross*1.45,fr"$t_\times={1e3*tx:.2f}$ ms",fontsize=7)
    a.set_yscale("log");a.set_xlim(0,5);a.set_ylim(1e-3,12)
    a.set_xlabel("time (ms)");a.set_ylabel(r"excess energy $\Delta E/(k_BT_b)$")
    a.set_title(title);a.legend(frameon=False);panel(a,label)

fig.savefig(OUT_FIG/"Figure4_thermal_no_crossing_vs_nonequilibrium_reproduced.pdf")
fig.savefig(OUT_FIG/"Figure4_thermal_no_crossing_vs_nonequilibrium_reproduced.png",dpi=300)
plt.show()


### What the four panels establish

- **Panels (a,b):** changing the temperature of a centered Gibbs state only rescales one positive relaxation law. The normalized collapse is the visual statement that a crossing is impossible anywhere within this preparation family.
- **Panel (c):** a displaced thermal state carries nonequilibrium spectral information in the first moment, so a higher-energy fast preparation can cross below a lower-energy slow preparation.
- **Panel (d):** a centered anisotropic Gaussian state can produce the same excess-energy crossing with zero mean displacement because the relevant spectral information is stored in the covariance.

The matched curves in (c) and (d) are intentional: the observable energy history can be the same even though the nonequilibrium information is stored in different statistical moments.


## 8. Exact-OU Monte Carlo check (optional teaching calculation)

The figure itself requires only moment propagation. The short simulation below verifies that the exact discrete OU step preserves the bath covariance without requiring a tiny Euler–Maruyama time step.


In [ ]:
RNG=np.random.default_rng(1234)
N=20_000
X=RNG.multivariate_normal(np.zeros(2),Sigma_b,size=N)
eta=RNG.multivariate_normal(np.zeros(2),Sigma_eta,size=N)
X1=X@Phi.T+eta
sample_cov=np.cov(X1,rowvar=False,bias=True)
print("max sample covariance deviation from I:",np.max(np.abs(sample_cov-np.eye(2))))


## Reader extensions

- Vary the two initial Gibbs temperatures: the normalized curves must still collapse.
- Vary the rate ratio while keeping the initial excess-energy ratio fixed and track the crossing time.
- Rotate the displaced mean or excess covariance away from a pure eigendirection and observe the emergence of mixed slow/fast tails.
- Compare a randomized initial ensemble followed by deterministic evolution with a continuously forced OU bath; the covariance equations differ by the diffusion term.
